# 4.3 Één verdeling, of meerdere groepen — pinguïns (penguins)

[04.2](04.2-long-tail%20jeffrey.ipynb) fitte een familie op taxiritten en de fitter was het eens met het onderliggende mechanisme: één proces, één lange staart, één familie die de uitzonderlijk dure ritten verklaart. Dit notebook behandelt het geval waarin de fitter het met *niets* eens kan worden — niet omdat de data vreemd is, maar omdat het simpelweg niet om één enkele populatie gaat. De vraag die hier beantwoord wordt: **wanneer een fit faalt, is de verdelingsfamilie verkeerd, of klopt de groepering van de data niet?** We gebruiken het gereedschap uit 04.2 — `DistributionFitter`, `fit_table` met haar twee winnaars, `PlotFits` en een qq-plot gericht op de staarten — en de mechanismen uit 04.1, met name de centrale limietstelling: een biologische meting die de som is van vele kleine groeibijdragen zou normaal verdeeld moeten zijn.

De data is de pinguïn-showcase uit [02.2](../lesson2/02.2-comparing_categories%20jeffrey.ipynb): drie soorten, twee geslachten, vier metingen per vogel. Les 2 vergeleek hun gemiddelden. Hier is de variabele **vlerklengte** (*flipper length*), en de eerste stap is bewust de verkeerde: we gooien alle vogels op één hoop (*pooling*) en fitten.


In [ ]:
import numpy as np
import pandas as pd

from goad_toolkit.analytics import DistributionFitter, FitResult, fit_table
from goad_toolkit.distributions import DistributionRegistry
from goad_toolkit.visualizer import (
    DistPlot,
    ECDFPlot,
    FitPlotSettings,
    HistogramPlot,
    PlotFits,
    PlotSettings,
    QQPlot,
)

from wa_analyzer.data import load_showcase

penguins = load_showcase("penguins").dropna(subset=["flipper_length_mm", "sex"]).reset_index(drop=True)
flipper = penguins["flipper_length_mm"].to_numpy().astype(float)
print(f"{len(penguins)} birds with a flipper measurement and a recorded sex")
print(f"mean: {flipper.mean():.1f} mm   median: {np.median(flipper):.1f} mm")

## 4.3.1 Alles samenvoegen (poolen) en fitten

Een vlerk groeit door vele kleine toenames tijdens de ontwikkeling van een vogel, en een som van vele onafhankelijke bijdragen is exact het mechanisme achter de normale verdeling (04.1 §4.1.2). De hypothese is dus `norm`, en de samengevoegde fit is de toets. Uit de printout blijkt nog niets verontrustends: gemiddelde en mediaan liggen dicht bij elkaar, wat precies is wat een symmetrische familie voorspelt. Het histogram vertelt echter een heel ander verhaal.


In [ ]:
fitter = DistributionFitter(DistributionRegistry(), seed=42)
flipper_fits = fitter.fit(flipper, discrete=False)

fig = PlotFits(PlotSettings(figsize=(12, 4), xlabel="flipper length (mm)", ylabel="density",
                            title="All 333 birds: the top three fits")).plot(
    data=flipper, fit_results=flipper_fits, fitplotsettings=FitPlotSettings(bins=30, max_fits=3),
)

In [ ]:
fit_table(flipper_fits)[["distribution", "log_likelihood", "ks_stat", "ks_pvalue", "best_likelihood", "best_ks"]]

Twee bulten: een grote rond de 190 mm, een kleinere rond de 215 mm, en een dal ertussenin waar nauwelijks een vogel zit. Geen enkele verdelingsfamilie in het register heeft twee bulten, dus elke gefitte curve is een compromis. De log-likelihood winnaar, `beta`, is het meest eerlijke compromis dat beschikbaar is — één brede, vlakke bult over beide toppen heen — en de KS-winnaar is weer een heel andere familie. In 04.2 waren beide winnaars het eens; hier zijn ze het oneens, *én de p-waardekolom verwerpt elke familie in de tabel*. Die combinatie is de meest informatieve uitkomst die `fit_table` kan opleveren, en het betekent allerminst dat het algoritme gefaald heeft. Maximum likelihood vond de best mogelijke curve door deze getallen; de getallen zijn alleen nooit door één enkele curve voortgebracht.

De qq-plot laat zien waar het compromis breekt, op een manier die een histogram niet kan tonen: het toont *welke* waarden de familie simpelweg niet kan accommoderen.


In [ ]:
best = next(f for f in flipper_fits if isinstance(f, FitResult) and f.best_likelihood)
norm_fit = fitter.fit_distribution("norm", flipper)

qq = PlotSettings(
    figsize=(10, 4.5),  # ty: ignore[invalid-argument-type]
    title="Pooled flipper length: the fitted family has no room for a gap",
    subplot_titles=[f"flipper vs. fitted {best.distribution}", "flipper vs. fitted normal"],
    xlabel="theoretical quantile",
    ylabel="flipper length (mm)",
    max_cols=2,
)
host = QQPlot(qq)
fig, axes = host.create_figure(n_plots=2)
host.plot_on_axes(QQPlot(qq), axes[0], data=flipper, distribution=best.frozen_dist)
_ = host.plot_on_axes(QQPlot(qq), axes[1], data=flipper, distribution=norm_fit.frozen_dist)  # ty: ignore[unresolved-attribute]

Beide panelen vertonen exact dezelfde **knik**. Door de concentratie vogels tussen ongeveer 190 en 205 mm vallen de punten steeds verder onder de referentielijn — er zijn hier meer vogels dan de familie verwacht — en vervolgens, net voorbij 205 mm, springen ze vrijwel verticaal over de lijn heen: een bereik van theoretische kwantielen waarvoor in de data geen enkele vogel bestaat. Dat is het dal in het histogram, van opzij bekeken. Een enkele verdelingsfamilie, ongeacht haar parameters, kan onmogelijk een leegte produceren midden in haar eigen domein. De fit liegt niet: de beta-verdeling is echt de beste curve door deze data. De *familie* is echter verkeerd voor de *populatie*, en wel omdat er hier helemaal geen sprake is van één populatie. Les 2 maakte al duidelijk wat de echte groepen zijn.


## 4.3.2 Uitsplitsen per soort: drie vormen

Dezelfde variabele, eenmaal per soort, op één as zonder verstorende bins — `ECDFPlot` accepteert een tweede steekproef als `compare`, en een derde kan worden toegevoegd via `plot_on`.


In [ ]:
by_species = {name: group["flipper_length_mm"].to_numpy().astype(float) for name, group in penguins.groupby("species")}

species_ecdf = PlotSettings(figsize=(9, 4), title="Flipper length per species: three curves, not one",
                            xlabel="flipper length (mm)", ylabel="share of birds at or below")
ecdf = ECDFPlot(species_ecdf)
fig, ax = ecdf.plot(data=by_species["Adelie"], compare=by_species["Chinstrap"], label="Adelie", compare_label="Chinstrap")
_ = ecdf.plot_on(ECDFPlot(species_ecdf), data=by_species["Gentoo"], label="Gentoo", color="darkgreen")

In [ ]:
def describe_fit(name: str, values: np.ndarray) -> list:
    """Fit every continuous family and report how the normal did against the field."""
    fits = fitter.fit(values, discrete=False)
    table = fit_table(fits)
    top = table["log_likelihood"].max()
    normal = table.loc[table["distribution"] == "norm"].iloc[0]
    within_three = int((table["log_likelihood"] > top - 3).sum())
    rejected = ", ".join(table.loc[table["ks_pvalue"] < 0.05, "distribution"]) or "none"
    print(f"{name:14s} n={len(values):3d}  mean {values.mean():6.1f}  sd {values.std(ddof=1):4.1f}  |  "
          f"norm: KS p={normal['ks_pvalue']:.2f}, {normal['log_likelihood'] - top:+.1f} from the winner;  "
          f"{within_three} families within 3 log-lik points;  rejected: {rejected}")
    return fits


species_hist = PlotSettings(
    figsize=(13, 3.5),  # ty: ignore[invalid-argument-type]
    title="Each species on its own, with its fitted normal",
    subplot_titles=list(by_species),
    xlabel="flipper length (mm)",
    max_cols=3,
    sharex=True,
)
host = HistogramPlot(species_hist)
fig, axes = host.create_figure(n_plots=3)
for ax, (name, values) in zip(axes, by_species.items()):
    describe_fit(name, values)
    host.plot_on_axes(HistogramPlot(species_hist), ax, data=values, bins=15, color="lightgrey")
    host.plot_on_axes(DistPlot(species_hist), ax, distribution=fitter.fit_distribution("norm", values).frozen_dist)  # ty: ignore[unresolved-attribute]
    ax.set_ylabel("")

Drie klokvormen. Adelie en Chinstrap overlappen grotendeels, met Chinstrap enkele millimeters naar rechts verschoven; Gentoo bevindt zich meer dan twintig millimeter verderop. De lege strook tussen de Chinstrap-curve en de Gentoo-curve in de ECDF verklaart het dal in het samengevoegde histogram en de knik in de qq-plot. Per soort is de fitter volkomen tevreden: de normale verdeling wordt voor geen van de drie verworpen. Bij Gentoo wint het of zit het binnen enkele log-likelihood punten van de top (vier punten achter een beta met twee extra parameters), en vijf of zes families bevinden zich binnen drie punten van de top. Die gelijkstand is de kernles: met ongeveer honderd vogels kan de data een normale verdeling niet onderscheiden van een Weibull of een scheve normale verdeling (*skew-normal*). Wanneer de data het onderscheid niet kan maken, is de familie die door het onderliggende mechanisme wordt voorspeld de beste keuze. De families die *wél* worden verworpen — uniform en exponentieel overal, gamma voor Adelie op het randje — worden om de juiste reden verworpen: een klokvorm is niet plat en start niet bij een harde wand.

De fit die faalde in §4.3.1 was dus geen tekortkoming van `norm`. Het was een fout in de observatie-eenheid: een "pinguïn" is geen biologische eenheid met een vaste vlerklengte; een *soort* pinguïn is dat wel.


## 4.3.3 Dezelfde stap nog een keer: Gentoo uitgesplitst naar geslacht

Een goedgekeurde fit is geen sluitend bewijs dat je een enkelvoudige homogene populatie hebt bereikt. Gentoo alleen doorstond elke bovenstaande controle, maar de gegroepeerde staafdiagrammen uit les 02.2 toonden al aan dat mannetjes en vrouwtjes verschillen in grootte. We splitsen nog een keer.


In [ ]:
gentoo = penguins[penguins["species"] == "Gentoo"]
by_sex = {sex: group["flipper_length_mm"].to_numpy().astype(float) for sex, group in gentoo.groupby("sex")}

describe_fit("Gentoo, pooled", by_species["Gentoo"])
for sex, values in by_sex.items():
    describe_fit(f"Gentoo {sex.lower()}", values)

sex_split = PlotSettings(
    figsize=(12, 4),
    title="Gentoo flipper length by sex",
    subplot_titles=["two histograms, two fitted normals", "the same two samples as ECDFs"],
    xlabel="flipper length (mm)",
    max_cols=2,
)
host = HistogramPlot(sex_split)
fig, axes = host.create_figure(n_plots=2)
for (sex, values), colour in zip(by_sex.items(), ["steelblue", "crimson"]):
    host.plot_on_axes(HistogramPlot(sex_split), axes[0], data=values, bins=12, color=colour, alpha=0.4)
    host.plot_on_axes(DistPlot(sex_split), axes[0], distribution=fitter.fit_distribution("norm", values).frozen_dist,  # ty: ignore[unresolved-attribute]
                      color=colour, label=sex.lower())
axes[0].legend()
_ = host.plot_on_axes(ECDFPlot(sex_split), axes[1], data=by_sex["Female"], compare=by_sex["Male"],
                      label="female", compare_label="male")

Twee klokvormen, negen millimeter uit elkaar, elk aanzienlijk smaller dan de verdeling waarin ze verstopt zaten: de samengevoegde Gentoo-standaarddeviatie van 6,6 mm krimpt naar 3,9 mm voor de vrouwtjes en 5,7 mm voor de mannetjes, en de normale verdeling wordt voor geen van beide verworpen (het histogram van de mannetjes is wat rafelig, maar dat is hoe 61 vogels gemeten op de millimeter eruitzien). De samengevoegde Gentoo-fit slaagde voor de KS-toets met p ≈ 0,2 en was al die tijd toch een mengsel. Dat is geen zwakte van de statistische toets: twee klokvormen die minder dan twee standaarddeviaties uit elkaar liggen, tellen op tot een vorm die zelf nagenoeg perfect klokvormig is. Geen enkel fit-algoritme kan daar doorheen kijken. Wat je vertelde om te splitsen was niet de tabel, maar domeinkennis van de data — de vraag uit les 02.2: *wat stelt één rij hier voor?*, en het antwoord: één vogel, van een specifieke soort én een specifiek geslacht.

Een mengsel verbergt haar subgroepen exact zoals het geaggregeerde gemiddelde uit [02.3](../lesson2/02.3-simpsons_paradox%20jeffrey.ipynb) subgroepen verborg: de samengevoegde samenvatting is rekenkundig correct, maar beschrijft niemand.


## 4.3.4 Een uitbijter is een uitspraak over een populatie

Reden 2 uit [04.1](04.1-parametric-distributions%20jeffrey.ipynb): een punt is uitsluitend ongebruikelijk *relatief ten opzichte van een verdeling*. Deze data heeft twee kandidaat-verdelingen voor elke vogel — de samengevoegde fit uit §4.3.1 en de normale verdeling voor zijn eigen soort en geslacht — en ze geven een totaal ander antwoord op dezelfde vraag. We beginnen met de vogel met de langste vlerk in de hele dataset.


In [ ]:
by_group = {
    key: fitter.fit_distribution("norm", group["flipper_length_mm"].to_numpy().astype(float)).frozen_dist  # ty: ignore[unresolved-attribute]
    for key, group in penguins.groupby(["species", "sex"])
}
pooled = best.frozen_dist

longest = penguins.loc[penguins["flipper_length_mm"].idxmax()]
value = float(longest["flipper_length_mm"])
own = by_group[(longest["species"], longest["sex"])]

pooled_tail = 1 - pooled.cdf(value)
own_tail = 1 - own.cdf(value)
print(f"longest flipper: {value:.0f} mm, a {longest['sex'].lower()} {longest['species']}")
print(f"P(flipper >= {value:.0f}) under the pooled {best.distribution}:  {pooled_tail:.4f}  (~1-in-{1 / pooled_tail:,.0f})")
print(f"    ...whose support ends at {pooled.support()[1]:.0f} mm: it puts zero probability on anything longer")
print(f"P(flipper >= {value:.0f}) under the {longest['species']} {longest['sex'].lower()} normal: {own_tail:.3f}  (~1-in-{1 / own_tail:,.0f})")

Onder de samengevoegde fit is dit een zeldzame vogel van 1 op de ruim honderd, en de gefitte beta-verdeling — een begrensde familie — gaat nog verder: die plaatst een hard plafond enkele millimeters erboven, met de claim dat geen enkele pinguïn een langere vlerk kan hebben. Onder de fit voor zijn eigen groep is het simpelweg een flink Gentoo-mannetje, het soort dat je bij 1 op de 20 vogels tegenkomt. Zelfde vogel, zelfde meting; het verschil zit puur in de populatie waar de vraag over gesteld wordt.

De tegenstelling werkt ook de andere kant op, en die richting is statistisch gezien veel gevaarlijker. Bereken voor elke vogel de tweezijdige overschrijdingskans (*tail probability*: het deel van de verdeling dat minstens zo ver van het centrum ligt) onder beide fits, en bekijk waar de twee antwoorden het meest uiteenlopen.


In [ ]:
def tail(dist, x: np.ndarray) -> np.ndarray:
    """Two-sided tail probability: the share of `dist` at least this far from its centre."""
    p = dist.cdf(x)
    return 2 * np.minimum(p, 1 - p)


flagged = penguins[["species", "sex", "flipper_length_mm"]].copy()
flagged["tail_pooled"] = tail(pooled, flipper)
flagged["tail_own_group"] = [
    tail(by_group[(species, sex)], value)
    for species, sex, value in zip(flagged["species"], flagged["sex"], flagged["flipper_length_mm"])
]
threshold = 0.01  # roughly the two-sided "z beyond 2.6" line, but under whichever family is asked

print(f"birds beyond the {threshold} tail: {int((flagged.tail_pooled < threshold).sum())} under the pooled fit, "
      f"{int((flagged.tail_own_group < threshold).sum())} under their own group's fit")
flagged["ratio"] = flagged["tail_pooled"] / flagged["tail_own_group"]
pd.concat([
    flagged.nlargest(3, "ratio").assign(verdict="ordinary pooled, unusual for its group"),
    flagged.nsmallest(3, "ratio").assign(verdict="unusual pooled, less so for its group"),
]).drop(columns="ratio").round(4)

De bovenste rijen zijn degene waar je scherp op moet zijn. Een Adelie-vrouwtje met een vlerk van 202 mm bevindt zich vrijwel exact op de samengevoegde mediaan — de overschrijdingskans beweert dat niets alledaagser kan zijn — terwijl ze tweeënhalve standaarddeviatie te groot is voor een Adelie-vrouwtje, een zeldzame 1-op-100 vogel binnen haar eigen groep! De samengevoegde fit ziet haar niet, omdat ze bij 202 mm opgaat in de massa van een andere soort. De onderste rijen tonen het omgekeerde: de kortste en langste vlerken lijken onder de samengevoegde fit veel zeldzamer dan ze in werkelijkheid zijn binnen hun eigen groep.

Geen van beide kolommen bevat een rekenfout. Ze beantwoorden simpelweg verschillende vragen — *hoe ongebruikelijk binnen alle pinguïns* versus *hoe ongebruikelijk voor een vrouwelijk Adelie* — en slechts één van die vragen gaat over een biologisch bestaande populatie. "Drie standaarddeviaties van het gemiddelde" draagt altijd een stilzwijgend *van wélke verdeling* met zich mee; hier was de samengevoegde verdeling een wiskundige fictie, en elke z-score die daarop gebaseerd werd was een uitspraak over die fictie. De verdelingsfamilie was prima; de groepering van de data was fout.

Wat je vervolgens *doet* met een gemarkeerde vogel kunnen de getallen niet beslissen: een invoerfout om te verwijderen, een vogel waarvan het geslacht verkeerd werd genoteerd, of een uitzonderlijk groot vrouwtje. De overschrijdingskans vertelt je over welke waarnemingen je vragen moet stellen; iemand met domeinkennis van het veldonderzoek geeft het antwoord.


## 4.3.5 Jouw beurt: samengevoegd versus per persoon

Jouw eigen chat is in exact dezelfde zin een mengsel: elke berekening van "berichten per dag" of "seconden tussen berichten" over de volledige export voegt mensen met totaal verschillende gewoonten samen. De methodologische stap is exact wat dit notebook driemaal heeft uitgevoerd — fit, ontdek dat de fit niet klopt voor de populatie, splits de data, en fit opnieuw — en de vraag die je beantwoordt is of de uitsplitsing de *familie* verandert of enkel de *parameters*.

Twee opties (kies er één, dat is voldoende):

- **Berichten per dag, samengevoegd versus per auteur.** Fit de discrete families op de dagelijkse telling over de hele chat, en vervolgens op de dagelijkse telling van elke auteur afzonderlijk. `nbinom` (negatief-binomiaal) zal geaggregeerd winnen (geen enkele chat kent een constant tempo) — de vraag is of de *dispersie* krimpt per persoon, of dat elk individu net zo grillig appt als de groep als geheel.
- **Pauzes binnen een conversatie-uitbarsting (burst), samengevoegd versus per auteur.** Knip de tussentijden af op één uur zoals 04.4 doet, fit de exponentiële verdeling (`exponential`) geaggregeerd en per auteur, en vergelijk de *schaal* (schaalparameter/gemiddelde pauze). Een geaggregeerde gemiddelde pauze van negentig seconden kan in werkelijkheid bestaan uit één persoon die na dertig seconden reageert en een ander die er driehonderd seconden over doet.

Het scriptrecept, met de functies uit dit notebook:

```python
own = load_own_chat()
own["timestamp"] = pd.to_datetime(own["timestamp"])
per_day = own.set_index("timestamp").resample("D").size().to_numpy().astype(float)
pooled = fit_table(fitter.fit(per_day, discrete=True))

for author, messages in own.groupby("author"):
    per_day = messages.set_index("timestamp").resample("D").size().to_numpy().astype(float)
    describe_fit(author, per_day)  # pas toe: discrete=True, en lees nbinom's dispersie af
```

Noteer de samengevoegde fit en de fits per persoon als volwaardige zinnen met de parameters erin vermeld, plus één zin over de vraag of het samengevoegde getal überhaupt iemand accuraat beschreef. Selecteer vervolgens een bericht of een dag die door de geaggregeerde fit als ongebruikelijk werd bestempeld, en bereken de score onder de eigen fit van die afzender, exact zoals §4.3.4 deed met het 202 mm Adelie-vrouwtje.

## Reflectievragen

1. Voer de samengevoegde fit in plaats daarvan uit op `body_mass_g` (lichaamsgewicht). Het histogram toont een lange staart, het gemiddelde ligt boven de mediaan en de fitter is tevreden met een scheve verdelingsfamilie — exact wat les 04.2 je leerde te interpreteren als een multiplicatief proces. Splits de data uit zoals dit notebook deed met vlerklengte en leg uit wat die "staart" in werkelijkheid was.
2. De samengevoegde fit voor Gentoo doorstond elke statistische toets, maar bestond in feite uit twee verschillende groepen. Aan welke voorwaarde moeten twee groepen voldoen om een mengsel zó effectief te verbergen, en wat had je moeten *weten* — niet berekenen — om dit te vermoeden?
3. Voor je eigen chat: noem één variabele waarin een fit over iedereen een bericht, een dag of een pauze als ongebruikelijk zou bestempelen, terwijl het volkomen alledaags is voor de specifieke persoon van wie het afkomstig is.

---

**Waar dit nu naartoe gaat.** Uitsplitsen op basis van een vooraf bekende categorie is de ene helft van de vergelijkingsmethode. De andere helft is uitsplitsen op basis van *tijd* — dezelfde verdelingsfamilie aan weerszijden van een gebeurtenis (event), en een nuldistributie om te bepalen of het verschil op toeval kan berusten. Dat is [04.4-before-and-after](04.4-before-and-after%20jeffrey.ipynb), eerst op de IRC-chat en vervolgens op je eigen WhatsApp-data.
